# Classification task with CWT and CNN model

This model is based on ["Real-Time Stress Detection via Photoplethysmogram Signals: Implementation of a Combined Continuous Wavelet Transform and Convolutional Neural Network on Resource-Constrained Microcontrollers"](https://ieeexplore.ieee.org/document/10668302) 

In [ ]:
import numpy as np
import pywt
import matplotlib.pyplot as plt
from scipy.signal import resample
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam

DATASET = "../../stressid-dataset/Physiological"
LABELS = "../labels.csv"
WINDOW_SIZE_SECS = 60


### Preprocessing functions

For window length: 10s
The samples per window if 64Hz would be 640 (64 x 10)
Consider 1 second stride as paper: stride samples would be 64

Maintaining the same window length of 10 seconds
Training dataset @500Hz: window(5000 samples), stride(500 samples)
Experiment unseen data @51.2Hz: window(512 samples), stride(51 samples)

To maintain the same image resolution
* a) Data could be downsampled 500Hz->100Hz or upsampled 51.2Hz->100Hz to achieve consistent 640 samples per window
* b) CWT generated images could be resized to a common resolution

In [ ]:
def segment_signal(
    signal: np.ndarray, fs: float, window_size_sec: float = 10, stride_sec: float = 1, target_fs: float | None = None
):
    num_samples = len(signal)
    if target_fs and target_fs != fs:
        target_num_samples = int(num_samples * target_fs / fs)
        signal = resample(signal, target_num_samples)
        fs = target_fs
        num_samples = len(signal)

    window_size = int(window_size_sec * fs)
    stride_size = int(stride_sec * fs)
    segments = []
    for start in range(0, num_samples - window_size, stride_size):
        segment = signal[start : start + window_size]
        segments.append(segment)
    return np.array(segments)


def cwt_transform(signal: np.ndarray, wavelet="morl", scales: list[int] = np.arange(1, 128)):
    coefficients, _ = pywt.cwt(signal, scales, wavelet)
    return coefficients


def create_cwt_images(segments: list[np.ndarray], img_size: tuple[int, int] = (128, 128)):
    images = []
    for seg in segments:
        cwt_img = cwt_transform(seg)
        cwt_img_resized = resize
        images.append(cwt_img)
    return np.array(images)


datagen = ImageDataGenerator(
    rescale=1.0 / 255, rotation_range=10, width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True
)

### CNN Model

In [ ]:
def build_model(input_shape):
    model = Sequential(
        [
            Conv2D(32, (3, 3), activation="relu", input_shape=input_shape),
            MaxPooling2D((2, 2)),
            Conv2D(64, (3, 3), activation="relu"),
            MaxPooling2D((2, 2)),
            Flatten(),
            Dense(128, activation="relu"),
            Dense(2, activation="softmax"),  # stress vs non-stress
        ]
    )
    model.compile(optimizer=Adam(), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


# Suppose you have: X_train (N, H, W), y_train
# Expand dims for channels
# X_train = np.expand_dims(X_train, -1)

# model = build_model(input_shape=(H, W, 1))
# history = model.fit(datagen.flow(X_train, y_train, batch_size=32), epochs=5)
